# Step 1: Multi-basin LSTM vs TFT on CAMELS-IND

Runs fully on Colab: code comes from GitHub, data is downloaded into this Colab session only. Nothing is saved on your laptop.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi -L || echo 'No GPU - training will be slow'
%cd /content
![ -d flood ] || git clone -q https://github.com/yatharth7115/flood.git
%cd /content/flood
!git pull -q && git log --oneline -1

In [ ]:
!pip install -q pytorch-forecasting pyyaml tabulate

## Settings
Set `QUICK_RUN = True` for a first test on 40 gauges (about 20–30 min). Set it to `False` for all gauges.

In [ ]:
import yaml
QUICK_RUN = True
SAVE_TO_DRIVE = True   # keeps results in Google Drive after the session ends

cfg = yaml.safe_load(open('configs/step1.yaml'))
cfg['data']['max_gauges'] = 40 if QUICK_RUN else None
if QUICK_RUN:
    cfg['lstm']['epochs'] = 10
    cfg['tft']['epochs'] = 8
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    cfg['results_dir'] = '/content/drive/MyDrive/flood_btp/results_step1'
yaml.safe_dump(cfg, open('configs/run.yaml', 'w'))
print('Results will be saved to', cfg['results_dir'])

## 1. Download CAMELS-IND v2.2 into Colab

In [ ]:
!python src/download.py configs/run.yaml

## 2. Look at the folder layout
If the next step fails, copy this output and share it so the loader can be adjusted.

In [ ]:
!python src/explore.py /content/data

## 3. Build the clean dataset

In [ ]:
!python src/dataset.py configs/run.yaml

## 4. Train the LSTM baseline

In [ ]:
!python src/train_lstm.py configs/run.yaml

## 5. Train the Temporal Fusion Transformer

In [ ]:
!python src/train_tft.py configs/run.yaml

## 6. Compare on the test period (2016–2020)

In [ ]:
!python src/evaluate.py configs/run.yaml
from IPython.display import Image, display
import os
R = cfg['results_dir']
for f in ['nse_cdf.png', 'hydrograph_focus.png', 'tft_variable_importance.png']:
    if os.path.exists(f'{R}/{f}'):
        display(Image(f'{R}/{f}', width=700))